# Validate Data With gx_framework

This notebook is the primary notebook entry point for the simplified Great Expectations framework.

It demonstrates:
- loading sample data into Spark
- validating a Spark DataFrame with `validate_dataframe(...)`
- validating a Spark table with `validate_table(...)`
- interpreting the returned result contract
- locating logs, saved validation outputs, and Delta-backed metrics
- trending retained metrics across multiple validation runs
- showing how failing validations persist failure metrics for later review

Prerequisites:
- a Spark session is available in the current notebook environment
- the repository `src/`, `gx/`, `config/`, and `logs/` folders are available
- the sample dataset and expectation suites in this repository are present

This notebook keeps framework logic in importable Python modules and uses notebook cells only for orchestration and explanation.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from pyspark.sql import SparkSession

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from gx_framework import validate_dataframe, validate_table

spark = SparkSession.getActiveSession() or SparkSession.builder.appName(
    "gx_framework_demo"
).getOrCreate()

DATASET_NAME = "green_tripdata_2017"
TABLE_NAME = "demo_green_tripdata_2017"
DATA_PATH = REPO_ROOT / "data" / "green_tripdata_2017_sample.csv"
LOGS_ROOT = REPO_ROOT / "logs"

print(f"Repository root: {REPO_ROOT}")
print(f"Sample data path: {DATA_PATH}")
print(f"Logs root: {LOGS_ROOT}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 14:37:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Repository root: /workspaces/great-expectations
Sample data path: /workspaces/great-expectations/data/green_tripdata_2017_sample.csv
Logs root: /workspaces/great-expectations/logs


## Load Sample Data

This section uses the repository sample CSV so the notebook exercises real project assets instead of synthetic data.

The dataset name `green_tripdata_2017` matches the mapping in `config/datasets.yml`, which resolves to the existing expectation suite `bronze.sales.green_tripdata_2017`.

In [2]:
df = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(DATA_PATH))
)

df.printSchema()
print(f"Row count: {df.count()}")
print(f"First 10 columns: {df.columns[:10]}")

root
 |-- vendorID: integer (nullable = true)
 |-- paymentType: integer (nullable = true)
 |-- passengerCount: integer (nullable = true)
 |-- tripDistance: double (nullable = true)
 |-- totalAmount: double (nullable = true)

Row count: 10
First 10 columns: ['vendorID', 'paymentType', 'passengerCount', 'tripDistance', 'totalAmount']


## Validate A DataFrame

Use the minimal public API when you already have a Spark DataFrame in memory.

This example relies on dataset-driven suite resolution, so the caller does not need to reference Great Expectations context objects, datasources, assets, or validators directly.

In [3]:
dataframe_result = validate_dataframe(
    df=df,
    dataset_name=DATASET_NAME,
    save_results=True,
    log_level="INFO",
)

print(json.dumps(dataframe_result, indent=2))

{"timestamp_utc": "2026-03-08T14:37:56.769549Z", "level": "INFO", "logger": "gx_framework.suite_resolver", "event": "suite_resolution", "taskName": "Task-49", "resolution_path": "config_mapping", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017"}
{"timestamp_utc": "2026-03-08T14:37:56.846676Z", "level": "INFO", "logger": "gx_framework", "event": "validation_started", "taskName": "Task-49", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_20260308_143756", "row_count": 10}
Calculating Metrics: 100%|██████████| 23/23 [00:01<00:00, 19.32it/s]
{"timestamp_utc": "2026-03-08T14:38:01.844607Z", "level": "INFO", "logger": "gx_framework", "event": "validation_result_saved", "taskName": "Task-49", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_20260308_143756", "result_path": "/workspaces/great-expectations/logs/

{
  "success": true,
  "dataset_name": "green_tripdata_2017",
  "suite_name": "bronze.sales.green_tripdata_2017",
  "run_name": "green_tripdata_2017_20260308_143756",
  "total_expectations": 3,
  "successful_expectations": 3,
  "failed_expectations": 0,
  "success_percent": 100.0,
  "validation_time_utc": "2026-03-08T14:38:01Z",
  "failure_details": []
}


## Validate A Table

Use `validate_table(...)` when the dataset is already registered in the active Spark session.

The function loads the table with `spark.table(table_name)` and delegates to the same validation flow used for DataFrames.

In [4]:
df.createOrReplaceTempView(TABLE_NAME)

table_result = validate_table(
    table_name=TABLE_NAME,
    dataset_name=DATASET_NAME,
    save_results=False,
    log_level="INFO",
)

print(json.dumps(table_result, indent=2))

{"timestamp_utc": "2026-03-08T14:38:04.971736Z", "level": "INFO", "logger": "gx_framework.suite_resolver", "event": "suite_resolution", "taskName": "Task-52", "resolution_path": "config_mapping", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017"}


{"timestamp_utc": "2026-03-08T14:38:05.046890Z", "level": "INFO", "logger": "gx_framework", "event": "validation_started", "taskName": "Task-52", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_20260308_143804", "row_count": 10}
26/03/08 14:38:05 WARN CacheManager: Asked to cache already cached data.
Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 53.10it/s]
{"timestamp_utc": "2026-03-08T14:38:05.548059Z", "level": "INFO", "logger": "gx_framework", "event": "validation_finished", "taskName": "Task-52", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "green_tripdata_2017_20260308_143804", "success": true, "failed_expectations": 0, "duration_seconds": 0.5763, "suite_path": "/workspaces/great-expectations/gx/expectations/bronze.sales.green_tripdata_2017.yml", "result_path": null}


{
  "success": true,
  "dataset_name": "green_tripdata_2017",
  "suite_name": "bronze.sales.green_tripdata_2017",
  "run_name": "green_tripdata_2017_20260308_143804",
  "total_expectations": 3,
  "successful_expectations": 3,
  "failed_expectations": 0,
  "success_percent": 100.0,
  "validation_time_utc": "2026-03-08T14:38:05Z",
  "failure_details": []
}


## Interpreting Results And Maintaining This Notebook

The returned dictionary is designed for notebooks and pipeline orchestration.

Key fields:
- `success` indicates whether the suite passed
- `suite_name` shows the resolved expectation suite
- `failed_expectations` and `failure_details` summarize validation issues
- `validation_time_utc` supports operational logging and audit trails

Operational notes:
- log files are written to `logs/gx_validation_YYYYMMDD.log`
- saved result payloads are written under `logs/validation_results/` when `save_results=True`
- for pipeline fail-fast behavior, call `validate_dataframe(..., fail_on_error=True)` or `validate_table(..., fail_on_error=True)`

Maintenance notes:
- restart the kernel if notebook imports become stale after local module edits
- keep framework logic in `src/` modules rather than expanding notebook utility code
- prefer the CSV-based demo path in this notebook if Delta support is unavailable in the current environment
- the notebooks in `archive/legacy-dq/notebooks/` are historical references and are no longer the primary documented path

In [5]:
summary = {
    "dataframe_success": dataframe_result["success"],
    "table_success": table_result["success"],
    "suite_name": dataframe_result["suite_name"],
    "failed_expectations": dataframe_result["failed_expectations"],
    "saved_result_files": sorted(
        path.name for path in (LOGS_ROOT / "validation_results").glob("*.json")
    ) if (LOGS_ROOT / "validation_results").exists() else [],
}

print(json.dumps(summary, indent=2))

if dataframe_result["failure_details"]:
    print("Failure details:")
    print(json.dumps(dataframe_result["failure_details"], indent=2))

{
  "dataframe_success": true,
  "table_success": true,
  "suite_name": "bronze.sales.green_tripdata_2017",
  "failed_expectations": 0,
  "saved_result_files": [
    "green_tripdata_2017_20260308_142253.json",
    "green_tripdata_2017_20260308_143756.json"
  ]
}


## Demonstrate Delta Metrics Logging With A Reusable Config

The active framework supports optional per-expectation metrics persistence to a Delta path.

This section uses the reusable example config at `config/validation_defaults.metrics_demo.yml`, runs a validation with metrics logging enabled, and then reads the Delta table back to confirm that validation metrics were captured.

In [9]:
import importlib
import shutil

import gx_framework.config as gx_config
import gx_framework.metrics_store as gx_metrics_store
import gx_framework.validator as gx_validator
from deltalake import DeltaTable

importlib.reload(gx_config)
importlib.reload(gx_metrics_store)
importlib.reload(gx_validator)

validate_dataframe = gx_validator.validate_dataframe

METRICS_CONFIG_PATH = REPO_ROOT / "config" / "validation_defaults.metrics_demo.yml"
METRICS_PATH = LOGS_ROOT / "notebook_demo_dq_metrics.delta"

if not METRICS_CONFIG_PATH.exists():
    raise FileNotFoundError(f"Metrics demo config not found: {METRICS_CONFIG_PATH}")

if METRICS_PATH.exists():
    shutil.rmtree(METRICS_PATH)

metrics_demo_result = validate_dataframe(
    df=df,
    dataset_name=DATASET_NAME,
    config_path=str(METRICS_CONFIG_PATH),
    run_name="metrics_demo_pass_01",
    save_results=False,
    log_level="INFO",
)

delta_table = DeltaTable(str(METRICS_PATH))
metrics_rows = delta_table.to_pyarrow_table().to_pylist()

metrics_demo_summary = {
    "validation_success": metrics_demo_result["success"],
    "config_path": str(METRICS_CONFIG_PATH),
    "metrics_store_path": str(METRICS_PATH),
    "delta_log_exists": (METRICS_PATH / "_delta_log").exists(),
    "metrics_rows_written": len(metrics_rows),
    "run_names": sorted({row["run_name"] for row in metrics_rows}),
    "expectation_types": [row["expectation_type"] for row in metrics_rows],
}

print(json.dumps(metrics_demo_summary, indent=2))

print("Sample metric row:")
print(json.dumps(metrics_rows[0], indent=2, default=str))

{"timestamp_utc": "2026-03-08T15:36:50.400740Z", "level": "INFO", "logger": "gx_framework.suite_resolver", "event": "suite_resolution", "taskName": "Task-73", "resolution_path": "config_mapping", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017"}


{"timestamp_utc": "2026-03-08T15:36:50.724691Z", "level": "INFO", "logger": "gx_framework", "event": "validation_started", "taskName": "Task-73", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "metrics_demo_pass_01", "row_count": 10}
26/03/08 15:36:50 WARN CacheManager: Asked to cache already cached data.
Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 38.21it/s]
{"timestamp_utc": "2026-03-08T15:36:51.478320Z", "level": "INFO", "logger": "gx_framework", "event": "validation_metrics_persisted", "taskName": "Task-73", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "metrics_demo_pass_01", "metrics_rows_written": 3, "metrics_store_path": "/workspaces/great-expectations/logs/notebook_demo_dq_metrics.delta", "metrics_store_format": "delta"}
{"timestamp_utc": "2026-03-08T15:36:51.480750Z", "level": "INFO", "logger": "gx_framework", "event": "validation_finished", "taskName": "

{
  "validation_success": true,
  "config_path": "/workspaces/great-expectations/config/validation_defaults.metrics_demo.yml",
  "metrics_store_path": "/workspaces/great-expectations/logs/notebook_demo_dq_metrics.delta",
  "delta_log_exists": true,
  "metrics_rows_written": 3,
  "run_names": [
    "metrics_demo_pass_01"
  ],
  "expectation_types": [
    "expect_table_row_count_to_be_between",
    "expect_column_values_to_not_be_null",
    "expect_column_values_to_be_in_set"
  ]
}
Sample metric row:
{
  "run_name": "metrics_demo_pass_01",
  "validation_time_utc": "2026-03-08T15:36:51Z",
  "dataset_name": "green_tripdata_2017",
  "suite_name": "bronze.sales.green_tripdata_2017",
  "run_success": true,
  "total_expectations": 3,
  "successful_expectations": 3,
  "failed_expectations": 0,
  "row_count": 10,
  "expectation_type": "expect_table_row_count_to_be_between",
  "column": null,
  "success": true,
  "unexpected_percent": null,
  "unexpected_count": null,
  "element_count": null,
  "

## Trend Metrics Across Multiple Runs

Delta persistence is append-only, so repeated validations accumulate a run history that can be queried for time-series analysis.

This section appends two more successful runs to the same metrics store and summarizes the retained metrics by run.

In [10]:
trend_run_names = ["metrics_demo_pass_02", "metrics_demo_pass_03"]

for run_name in trend_run_names:
    validate_dataframe(
        df=df,
        dataset_name=DATASET_NAME,
        config_path=str(METRICS_CONFIG_PATH),
        run_name=run_name,
        save_results=False,
        log_level="INFO",
    )

metrics_history_rows = DeltaTable(str(METRICS_PATH)).to_pyarrow_table().to_pylist()
pass_history_rows = [
    row
    for row in metrics_history_rows
    if str(row["run_name"]).startswith("metrics_demo_pass_")
]

metrics_trend_summary = []
for run_name in sorted({row["run_name"] for row in pass_history_rows}):
    run_rows = [row for row in pass_history_rows if row["run_name"] == run_name]
    metrics_trend_summary.append(
        {
            "run_name": run_name,
            "validation_time_utc": min(row["validation_time_utc"] for row in run_rows),
            "run_success": bool(run_rows[0]["run_success"]),
            "row_count": run_rows[0]["row_count"],
            "evaluated_expectations": len(run_rows),
            "failed_expectations": run_rows[0]["failed_expectations"],
            "failed_expectation_types": sorted(
                {row["expectation_type"] for row in run_rows if not row["success"]}
            ),
        }
    )

print(json.dumps(metrics_trend_summary, indent=2))

{"timestamp_utc": "2026-03-08T15:36:56.230221Z", "level": "INFO", "logger": "gx_framework.suite_resolver", "event": "suite_resolution", "taskName": "Task-76", "resolution_path": "config_mapping", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017"}
{"timestamp_utc": "2026-03-08T15:36:56.281041Z", "level": "INFO", "logger": "gx_framework", "event": "validation_started", "taskName": "Task-76", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "metrics_demo_pass_02", "row_count": 10}
26/03/08 15:36:56 WARN CacheManager: Asked to cache already cached data.
Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 63.79it/s] 
{"timestamp_utc": "2026-03-08T15:36:56.761789Z", "level": "INFO", "logger": "gx_framework", "event": "validation_metrics_persisted", "taskName": "Task-76", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "metrics_demo_pass_02", "me

[
  {
    "run_name": "metrics_demo_pass_01",
    "validation_time_utc": "2026-03-08T15:36:51Z",
    "run_success": true,
    "row_count": 10,
    "evaluated_expectations": 3,
    "failed_expectations": 0,
    "failed_expectation_types": []
  },
  {
    "run_name": "metrics_demo_pass_02",
    "validation_time_utc": "2026-03-08T15:36:56Z",
    "run_success": true,
    "row_count": 10,
    "evaluated_expectations": 3,
    "failed_expectations": 0,
    "failed_expectation_types": []
  },
  {
    "run_name": "metrics_demo_pass_03",
    "validation_time_utc": "2026-03-08T15:36:57Z",
    "run_success": true,
    "row_count": 10,
    "evaluated_expectations": 3,
    "failed_expectations": 0,
    "failed_expectation_types": []
  }
]


## Demonstrate A Failing Validation With Persisted Failure Metrics

The same Delta store also retains failed runs. This section injects known bad values into columns covered by the expectation suite, runs validation without fail-fast behavior, and then inspects the persisted failed metric rows.

In [11]:
from pyspark.sql import functions as F

vendor_id_type = next(
    field.dataType for field in df.schema.fields if field.name == "vendorID"
 )
payment_type = next(
    field.dataType for field in df.schema.fields if field.name == "paymentType"
 )

failing_df = (
    df.withColumn("vendorID", F.lit(None).cast(vendor_id_type))
    .withColumn("paymentType", F.lit(99).cast(payment_type))
)

failing_metrics_result = validate_dataframe(
    df=failing_df,
    dataset_name=DATASET_NAME,
    config_path=str(METRICS_CONFIG_PATH),
    run_name="metrics_demo_fail_01",
    save_results=False,
    log_level="INFO",
)

metrics_history_rows = DeltaTable(str(METRICS_PATH)).to_pyarrow_table().to_pylist()
failure_history_rows = [
    row for row in metrics_history_rows if row["run_name"] == "metrics_demo_fail_01" 
]
failed_metric_rows = [row for row in failure_history_rows if not row["success"]]

failure_metrics_summary = {
    "validation_success": failing_metrics_result["success"],
    "failed_expectations": failing_metrics_result["failed_expectations"],
    "persisted_metric_rows": len(failure_history_rows),
    "persisted_failed_metric_rows": len(failed_metric_rows),
    "failed_expectation_types": sorted(
        {row["expectation_type"] for row in failed_metric_rows}
    ),
    "unexpected_counts": [row["unexpected_count"] for row in failed_metric_rows],
}

print(json.dumps(failure_metrics_summary, indent=2))

print("Sample persisted failure metric:")
print(
    json.dumps(
        failed_metric_rows[0] if failed_metric_rows else failure_history_rows[0],
        indent=2,
        default=str,
    )
)

{"timestamp_utc": "2026-03-08T15:37:00.383629Z", "level": "INFO", "logger": "gx_framework.suite_resolver", "event": "suite_resolution", "taskName": "Task-79", "resolution_path": "config_mapping", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017"}
{"timestamp_utc": "2026-03-08T15:37:00.431921Z", "level": "INFO", "logger": "gx_framework", "event": "validation_started", "taskName": "Task-79", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "metrics_demo_fail_01", "row_count": 10}
Calculating Metrics: 100%|██████████| 23/23 [00:00<00:00, 54.66it/s] 
{"timestamp_utc": "2026-03-08T15:37:01.010706Z", "level": "INFO", "logger": "gx_framework", "event": "validation_metrics_persisted", "taskName": "Task-79", "dataset_name": "green_tripdata_2017", "suite_name": "bronze.sales.green_tripdata_2017", "run_name": "metrics_demo_fail_01", "metrics_rows_written": 3, "metrics_store_path": "/workspaces/great-expectat

{
  "validation_success": false,
  "failed_expectations": 2,
  "persisted_metric_rows": 3,
  "persisted_failed_metric_rows": 2,
  "failed_expectation_types": [
    "expect_column_values_to_be_in_set",
    "expect_column_values_to_not_be_null"
  ],
  "unexpected_counts": [
    10,
    10
  ]
}
Sample persisted failure metric:
{
  "run_name": "metrics_demo_fail_01",
  "validation_time_utc": "2026-03-08T15:37:00Z",
  "dataset_name": "green_tripdata_2017",
  "suite_name": "bronze.sales.green_tripdata_2017",
  "run_success": false,
  "total_expectations": 3,
  "successful_expectations": 1,
  "failed_expectations": 2,
  "row_count": 10,
  "expectation_type": "expect_column_values_to_not_be_null",
  "column": "vendorID",
  "success": false,
  "unexpected_percent": 100.0,
  "unexpected_count": 10,
  "element_count": 10,
  "details_json": "{\"success\": false, \"expectation_config\": {\"type\": \"expect_column_values_to_not_be_null\", \"kwargs\": {\"batch_id\": \"fabric_spark_datasource-green_t